In [4]:
import tkinter as tk
from tkinter import messagebox
import pymysql

class DictionaryApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Python 수업 용어 사전")
        self.root.geometry("400x500")
        a
        # DB 연결 정보 (사용자 입력으로 변경됨)
        self.db_host = "127.0.0.1"
        self.connection = None
        
        # 초기 로그인 화면 구성
        self.login_frame = tk.Frame(self.root)
        self.login_frame.pack(pady=50)
        
        tk.Label(self.login_frame, text="DB 계정 ID:").pack()
        self.ent_user = tk.Entry(self.login_frame)
        self.ent_user.pack()
        
        tk.Label(self.login_frame, text="Password:").pack()
        self.ent_pw = tk.Entry(self.login_frame, show="*")
        self.ent_pw.pack()
        
        tk.Button(self.login_frame, text="DB 접속", command=self.connect_db).pack(pady=10)

    def connect_db(self):
        """MySQL 서버에 접속 시도"""
        user_id = self.ent_user.get()
        user_pw = self.ent_pw.get()
        
        try:
            self.connection = pymysql.connect(
                host=self.db_host,
                user=user_id,
                password=user_pw,
                db='class_dict_db', # 미리 생성한 DB 이름
                charset='utf8mb4',
                cursorclass=pymysql.cursors.DictCursor
            )
            messagebox.showinfo("성공", "데이터베이스에 연결되었습니다.")
            self.show_search_ui()
        except Exception as e:
            messagebox.showerror("오류", f"접속 실패: {e}")

    def show_search_ui(self):
        """로그인 성공 후 검색 화면"""
        self.login_frame.destroy() # 로그인 창 제거
        
        search_frame = tk.Frame(self.root)
        search_frame.pack(pady=20, fill="both", expand=True)
        
        tk.Label(search_frame, text="검색할 용어를 입력하세요:").pack()
        self.ent_search = tk.Entry(search_frame, width=30)
        self.ent_search.pack(pady=5)
        
        tk.Button(search_frame, text="검색", command=self.search_word).pack()
        
        self.txt_result = tk.Text(search_frame, height=15, width=45)
        self.txt_result.pack(pady=20)

    def search_word(self):
        """DB에서 단어 검색"""
        word = self.ent_search.get()
        if not word:
            return

        try:
            with self.connection.cursor() as cursor:
                # SQL 인젝션 방지를 위해 파라미터화된 쿼리 사용
                sql = "SELECT definition FROM terms WHERE word = %s"
                cursor.execute(sql, (word,))
                result = cursor.fetchone()
                
                self.txt_result.delete(1.0, tk.END) # 이전 내용 삭제
                if result:
                    self.txt_result.insert(tk.END, f"[{word}]의 뜻:\n\n{result['definition']}")
                else:
                    self.txt_result.insert(tk.END, "검색 결과가 없습니다.")
        except Exception as e:
            messagebox.showerror("오류", f"조회 중 오류 발생: {e}")

if __name__ == "__main__":
    root = tk.Tk()
    app = DictionaryApp(root)
    root.mainloop()